In [7]:
import pandas as pd

def get_load_profile(load_path, desired_timestep_minutes):
    """
    Get data out of a CSV file
    Original ResStock data is 15 minute kWh data - need hourly for TMY comparison and to convert to kW
    """
    df = pd.read_csv(load_path)
    timeseries = df["out.electricity.total.energy_consumption"].values
    
    if desired_timestep_minutes < 15:
        raise ValueError("get_load_profile is not set up for timesteps less than 15 minutes.")
    elif desired_timestep_minutes == 15:
        return timeseries * 4 # Convert from kWh to kW
    elif desired_timestep_minutes % 15 != 0: 
        raise ValueError("get_load_profile is not set up for that aren't evenly divisiable by 15. Pick 15, 30, or 60.")
    else:
        averaged_timesteps = []
        steps_per_step = desired_timestep_minutes / 15
        steps_per_hr = 60 / desired_timestep_minutes
        step = 0
        avg_kwhs = 0
        for kwh in timeseries:
             avg_kwhs += kwh
             step += 1
             if step == steps_per_step:
                 averaged_timesteps.append(avg_kwhs * steps_per_hr)
                 step = 0
                 avg_kwhs = 0
        return averaged_timesteps

In [13]:
import csv
import os

file_dir = os.path.abspath('')
print(file_dir)

load_path = file_dir + "/load_data/load-data-AZ-112157-0.csv"

load = get_load_profile(load_path, 60)

filename = 'load-data-AZ-112157_hourly.csv'

epw_file = open(filename, 'w', newline='\n', encoding='utf-8')
writer = csv.writer(epw_file, delimiter=',', lineterminator='\n')
writer.writerow(['kW'])

for l in load:
    writer.writerow([str(l)])

epw_file.close()

/Users/bspeetle/Desktop/repo/SAM-analyses/2025/battwatts_sensitivity
